<a href="https://colab.research.google.com/github/Pulakspbl/Machine-learning-in-Python-with-scikit-learn/blob/main/Model_6.1_Ensemble/Model_6_1_Ensemble_Exercise_M6_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The aim of this exercise is to:

* verifying if a random forest or a gradient-boosting decision tree overfit if
  the number of estimators is not properly chosen;
* use the early-stopping strategy to avoid adding unnecessary trees, to get
  the best generalization performances.

We use the California housing dataset to conduct our experiments.

In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

data, target = fetch_california_housing(return_X_y=True, as_frame=True)
target *= 100  # rescale the target in k$
data_train, data_test, target_train, target_test = train_test_split(
    data, target, random_state=0, test_size=0.5
)

Create a gradient boosting decision tree with max_depth=5 and learning_rate=0.5

In [2]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_validate

gradient_boosting = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=5, learning_rate=0.5)

cv_results_gbdt = cross_validate(
    gradient_boosting,
    data,
    target,
    scoring="neg_mean_absolute_error",
    n_jobs=2,
)

Also create a random forest with fully grown trees by setting max_depth=None.


In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate

random_forest = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    random_state=0,
    n_jobs=2
)

cv_results_rf = cross_validate(
    random_forest,
    data,
    target,
    scoring="neg_mean_absolute_error",
    n_jobs=2
)


For both the gradient-boosting and random forest models, create a validation
curve using the training set to assess the impact of the number of trees on
the performance of each model. Evaluate the list of parameters `param_range =
np.array([1, 2, 5, 10, 20, 50, 100])` and use the mean absolute error.

In [4]:
import numpy as np
from sklearn.model_selection import validation_curve
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

param_range = np.array([1, 2, 5, 10, 20, 50, 100])

# Gradient Boosting validation curve
train_scores_gbdt, test_scores_gbdt = validation_curve(
    GradientBoostingRegressor(max_depth=5, learning_rate=0.5),
    data_train,
    target_train,
    param_name="n_estimators",
    param_range=param_range,
    scoring="neg_mean_absolute_error",
    n_jobs=2,
)

# Random Forest validation curve
train_scores_rf, test_scores_rf = validation_curve(
    RandomForestRegressor(max_depth=None, random_state=0),
    data_train,
    target_train,
    param_name="n_estimators",
    param_range=param_range,
    scoring="neg_mean_absolute_error",
    n_jobs=2,
)

Both gradient boosting and random forest models improve when increasing the
number of trees in the ensemble. However, the scores reach a plateau where
adding new trees just makes fitting and scoring slower.

To avoid adding new unnecessary tree, unlike random-forest gradient-boosting
offers an early-stopping option. Internally, the algorithm uses an
out-of-sample set to compute the generalization performance of the model at
each addition of a tree. Thus, if the generalization performance is not
improving for several iterations, it stops adding trees.

Now, create a gradient-boosting model with `n_estimators=1_000`. This number
of trees is certainly too large. Change the parameter `n_iter_no_change` such
that the gradient boosting fitting stops after adding 5 trees that do not
improve the overall generalization performance.

In [5]:
gradient_boosting = GradientBoostingRegressor(
    n_estimators=1_000,
    max_depth=5,
    learning_rate=0.5,
    n_iter_no_change=5,
    random_state=0
)

Estimate the generalization performance of this model again using the
`sklearn.metrics.mean_absolute_error` metric but this time using the test set
that we held out at the beginning of the notebook. Compare the resulting value
with the values observed in the validation curve.

In [6]:
from sklearn.metrics import mean_absolute_error

gradient_boosting.fit(data_train, target_train)

target_pred = gradient_boosting.predict(data_test)

mae_test = mean_absolute_error(target_test, target_pred)

print("Test MAE:", mae_test)

Test MAE: 35.46129081606355
